## A. Model Training

### ✅ 1. Setup + Load Data

In [2]:
#!pip install mlflow joblib
#!pip install --upgrade pandas pyarrow fastparquet
import pandas as pd
import numpy as np
import joblib
import mlflow
import mlflow.sklearn
import xgboost as xgb
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# Load dataset
train_df = pd.read_parquet("../../data/merged_data/train.parquet")
val_df = pd.read_parquet("../../data/merged_data/val.parquet")

### ✅ Separate Features and Target.

In [3]:
# Separate features and target
X_train = train_df.drop(columns=['salary'])
y_train = train_df['salary']
X_val = val_df.drop(columns=['salary'])
y_val = val_df['salary']

# Storage for comparison table
results = []

### ✅ 2. Model Training using Linear Regression, XGBoost with Grid Search & Neural Network.

In [4]:
def evaluate_and_log(name, model, X, y_true, params=None):
    preds = model.predict(X)
    rmse = np.sqrt(mean_squared_error(y_true, preds))
    mae = mean_absolute_error(y_true, preds)
    r2 = r2_score(y_true, preds)
    
    # Store results for the summary table
    results.append({"Model": name, "RMSE": rmse, "MAE": mae, "R2": r2})
    
    # MLflow Logging
    with mlflow.start_run(run_name=name):
        if params:
            mlflow.log_params(params)
        mlflow.log_metrics({"rmse": rmse, "mae": mae, "r2": r2})
        mlflow.sklearn.log_model(model, name)
    
    print(f"{name} -> RMSE: {rmse:.2f}, MAE: {mae:.2f}, R2: {r2:.2f}")
    return rmse

# --- Model 1: Baseline (Linear Regression) ---
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
evaluate_and_log("Linear_Regression_Baseline", lr_model, X_val, y_val)

# --- Model 2: Tree Ensemble (XGBoost with Grid Search) ---
xgb_regressor = xgb.XGBRegressor(objective='reg:squarederror', random_state=42)
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5],
    'learning_rate': [0.1, 0.2]
}
grid_search = GridSearchCV(xgb_regressor, param_grid, cv=3, scoring='neg_mean_squared_error', n_jobs=-1)
grid_search.fit(X_train, y_train)

best_xgb = grid_search.best_estimator_
evaluate_and_log("Tuned_XGBoost", best_xgb, X_val, y_val, params=grid_search.best_params_)

# --- Model 3: Neural Network (MLP) ---
fast_nn = Pipeline([
    ('scaler', StandardScaler()),
    ('nn', MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=50, early_stopping=True, random_state=42))
])
fast_nn.fit(X_train, y_train)
evaluate_and_log("Neural_Network", fast_nn, X_val, y_val)

2026/04/24 22:46:01 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/04/24 22:46:01 INFO mlflow.store.db.utils: Updating database tables
2026/04/24 22:46:02 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/24 22:46:02 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/24 22:46:05 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


Linear_Regression_Baseline -> RMSE: 8114.77, MAE: 6245.39, R2: 0.96


2026/04/24 22:46:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/24 22:46:12 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/24 22:46:14 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


Tuned_XGBoost -> RMSE: 5213.35, MAE: 4161.21, R2: 0.98


/Users/anthonyjairam/Developer/COMP6940-Project/.venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (50) reached and the optimization hasn't converged yet.
  warnings.warn(
2026/04/24 22:46:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/24 22:46:22 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/24 22:46:24 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


Neural_Network -> RMSE: 5541.92, MAE: 4408.86, R2: 0.98


np.float64(5541.919479328314)

### ✅ 3. Creating Comparison Table & Selecting Best Model.

In [5]:
# Create Comparison Table
comparison_df = pd.DataFrame(results)
print("\n--- Model Comparison Summary ---")
print(comparison_df)

# Select Champion (lowest RMSE)
champion_row = comparison_df.loc[comparison_df['RMSE'].idxmin()]
champion_name = champion_row['Model']
print(f"\nChampion Model: {champion_name} (RMSE: {champion_row['RMSE']:.2f})")

# Save Champion Model to artifacts/
if champion_name == "Tuned_XGBoost":
    champion_model = best_xgb
elif champion_name == "Neural_Network":
    champion_model = fast_nn
else:
    champion_model = lr_model

joblib.dump(champion_model, "../../artifacts/champion_model.pkl")


--- Model Comparison Summary ---
                        Model         RMSE          MAE        R2
0  Linear_Regression_Baseline  8114.772758  6245.386901  0.955076
1               Tuned_XGBoost  5213.346334  4161.211914  0.981458
2              Neural_Network  5541.919479  4408.861378  0.979047

Champion Model: Tuned_XGBoost (RMSE: 5213.35)


['../../artifacts/champion_model.pkl']

### ✅ 4. Residuals.

In [6]:
# Generate Residuals
val_preds = champion_model.predict(X_val)
residuals_df = pd.DataFrame({
    'actual': y_val,
    'predicted': val_preds,
    'residual': val_preds - y_val
})

# Add country back for analysis in NB5
if 'country' in val_df.columns:
    residuals_df['country'] = val_df['country'].values
else:
    # If country was encoded, you might need to map it back or use the original index
    print("Warning: 'country' column not found in validation features.")

# Export to parquet
residuals_df.to_parquet("../../data/merged_data/residuals.parquet")
print("Residuals saved to ../../data/merged_data/residuals.parquet")

Residuals saved to ../../data/merged_data/residuals.parquet
